# YouTube Title Extractor & Generator

Extract titles from top-performing YouTube videos, analyze what works, and generate optimized titles — no coding required.

## How to use this tool

1. **Run Step 1** (click the play button) — installs the scraper. Only needed once.
2. **Run Step 2** — fill in the form to scrape YouTube titles for your keyword.
3. **Run Step 3** — analyzes title patterns and generates new optimized titles.
4. **(Optional) Step 4** — download results as CSV.

---

## Step 1: Setup (run this once)
Click the **play button** on the left side of the cell below. Wait until it says **"Ready!"**

In [ ]:
# ============================
# STEP 1: SETUP (run once)
# ============================
!pip install -q yt-dlp

import subprocess
import json
import csv
import os
from datetime import datetime
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML, clear_output
from google.colab import files as colab_files


def format_number(n):
    if n is None:
        return "N/A"
    if n >= 1_000_000_000:
        return f"{n / 1_000_000_000:.1f}B"
    if n >= 1_000_000:
        return f"{n / 1_000_000:.1f}M"
    if n >= 1_000:
        return f"{n / 1_000:.1f}K"
    return str(n)


def format_duration(seconds):
    if seconds is None:
        return "N/A"
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


def format_date(date_str):
    if not date_str or len(date_str) != 8:
        return "N/A"
    return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"


def run_yt_dlp(args):
    cmd = ["yt-dlp", "--dump-json", "--no-download", "--no-warnings"] + args
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if result.returncode == 0 and result.stdout.strip():
            entries = []
            for line in result.stdout.strip().split("\n"):
                line = line.strip()
                if line:
                    try:
                        entries.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
            return entries
    except subprocess.TimeoutExpired:
        print("  Request timed out, try again.")
    except Exception as e:
        print(f"  Error: {e}")
    return []


def extract_video_info(entry):
    return {
        "title": entry.get("title", "N/A"),
        "url": entry.get("webpage_url") or entry.get("url") or entry.get("original_url", "N/A"),
        "video_id": entry.get("id", "N/A"),
        "channel": entry.get("channel") or entry.get("uploader", "N/A"),
        "channel_subscribers": entry.get("channel_follower_count"),
        "views": entry.get("view_count"),
        "likes": entry.get("like_count"),
        "comments": entry.get("comment_count"),
        "duration": entry.get("duration"),
        "upload_date": entry.get("upload_date"),
    }


def sort_videos(videos, sort_by):
    if sort_by == "Most Views":
        return sorted(videos, key=lambda v: v.get("views") or 0, reverse=True)
    elif sort_by == "Newest First":
        return sorted(videos, key=lambda v: v.get("upload_date") or "0", reverse=True)
    elif sort_by == "Most Engagement (likes+comments)":
        def engagement(v):
            likes = v.get("likes") or 0
            comments = v.get("comments") or 0
            views = v.get("views") or 1
            return (likes + comments * 3) / views
        return sorted(videos, key=engagement, reverse=True)
    elif sort_by == "Most Subscribers":
        return sorted(videos, key=lambda v: v.get("channel_subscribers") or 0, reverse=True)
    return videos


def show_results_html(videos, titles_only=False):
    if not videos:
        display(HTML("<h3 style='color:red;'>No videos found. Try a different input.</h3>"))
        return

    if titles_only:
        html = f"<h3>Titles ({len(videos)} videos)</h3><ol>"
        for v in videos:
            html += f"<li style='margin-bottom:4px; font-size:14px;'><b>{v['title']}</b></li>"
        html += "</ol>"
        display(HTML(html))
        return

    html = f"<h3>Results ({len(videos)} videos)</h3>"
    html += "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
    html += "<tr style='background:#f0f0f0; text-align:left;'>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>#</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Title</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Channel</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Views</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Likes</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Duration</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Uploaded</th>"
    html += "<th style='padding:8px; border:1px solid #ddd;'>Subs</th>"
    html += "</tr>"

    for i, v in enumerate(videos, 1):
        bg = '#ffffff' if i % 2 == 1 else '#f9f9f9'
        url = v.get('url', '#')
        html += f"<tr style='background:{bg};'>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{i}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd;'><a href='{url}' target='_blank'>{v['title']}</a></td>"
        html += f"<td style='padding:6px; border:1px solid #ddd;'>{v['channel']}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['views'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['likes'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{format_duration(v['duration'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:center;'>{format_date(v['upload_date'])}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(v['channel_subscribers'])}</td>"
        html += "</tr>"

    html += "</table>"
    display(HTML(html))


def export_csv_and_download(videos, filename="youtube_titles.csv"):
    if not videos:
        print("No data to export.")
        return
    fieldnames = ["rank", "title", "channel", "subscribers", "views", "likes",
                  "comments", "duration_seconds", "upload_date", "url"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for i, v in enumerate(videos, 1):
            writer.writerow({
                "rank": i, "title": v["title"], "channel": v["channel"],
                "subscribers": v["channel_subscribers"] or "",
                "views": v["views"] or "", "likes": v["likes"] or "",
                "comments": v["comments"] or "",
                "duration_seconds": v["duration"] or "",
                "upload_date": format_date(v["upload_date"]), "url": v["url"],
            })
    colab_files.download(filename)
    print(f"Downloading {filename}...")


print("\n" + "=" * 50)
print("  Setup complete! Ready to use.")
print("=" * 50)
print("\nNow go to Step 2 below.")

---
## Step 2: Extract YouTube Titles

1. **Fill in the form** on the right side (click the form fields)
2. **Click the play button** to run

**Choose your mode:**
- `Video URL` — paste a direct YouTube video link
- `Search Results URL` — paste a YouTube search page URL
- `Keyword Search` — just type a topic/keyword


In [ ]:
# =============================================
# STEP 2: FILL IN THE FORM AND CLICK PLAY
# =============================================

#@title  { run: "auto", display-mode: "form" }

#@markdown ### Choose your mode and enter your input:

mode = "Keyword Search"  #@param ["Video URL", "Search Results URL", "Keyword Search"]
your_input = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Options:
sort_by = "Relevance (YouTube default)"  #@param ["Relevance (YouTube default)", "Most Views", "Newest First", "Most Engagement (likes+comments)", "Most Subscribers"]
number_of_results = 30  #@param {type:"slider", min:5, max:50, step:5}
show_titles_only = False  #@param {type:"boolean"}

# ---- Processing ----
if not your_input.strip():
    display(HTML("<h3 style='color:orange;'>Please enter a URL or keyword in the form above.</h3>"))
else:
    input_text = your_input.strip()
    videos = []

    if mode == "Video URL":
        print(f"Fetching title for: {input_text}")
        # Support multiple URLs separated by commas or newlines
        urls = [u.strip() for u in input_text.replace(",", "\n").split("\n") if u.strip()]
        for url in urls:
            print(f"  Fetching: {url}")
            entries = run_yt_dlp([url])
            for entry in entries:
                videos.append(extract_video_info(entry))

    elif mode == "Search Results URL":
        parsed = urlparse(input_text)
        params = parse_qs(parsed.query)
        query = params.get("search_query", [None])[0]
        if not query:
            if "/hashtag/" in parsed.path:
                query = parsed.path.split("/hashtag/")[-1]
        if not query:
            display(HTML("<h3 style='color:red;'>Could not extract search query from that URL.</h3>"))
            display(HTML("<p>Expected format: <code>https://www.youtube.com/results?search_query=your+search</code></p>"))
        else:
            print(f"Searching YouTube for: \"{query}\"")
            print(f"Fetching top {number_of_results} results...")
            search_entries = run_yt_dlp([f"ytsearch{number_of_results}:{query}", "--flat-playlist"])
            if search_entries:
                print(f"Found {len(search_entries)} results. Getting details...")
                for entry in search_entries:
                    vid_id = entry.get("id") or entry.get("url")
                    if vid_id:
                        url = vid_id if vid_id.startswith("http") else f"https://www.youtube.com/watch?v={vid_id}"
                        details = run_yt_dlp([url])
                        for d in details:
                            videos.append(extract_video_info(d))
                            print(f"  [{len(videos)}/{len(search_entries)}] {extract_video_info(d)['title'][:60]}")

    elif mode == "Keyword Search":
        print(f"Searching YouTube for: \"{input_text}\"")
        print(f"Fetching top {number_of_results} results...")
        search_entries = run_yt_dlp([f"ytsearch{number_of_results}:{input_text}", "--flat-playlist"])
        if search_entries:
            print(f"Found {len(search_entries)} results. Getting details...")
            for entry in search_entries:
                vid_id = entry.get("id") or entry.get("url")
                if vid_id:
                    url = vid_id if vid_id.startswith("http") else f"https://www.youtube.com/watch?v={vid_id}"
                    details = run_yt_dlp([url])
                    for d in details:
                        videos.append(extract_video_info(d))
                        print(f"  [{len(videos)}/{len(search_entries)}] {extract_video_info(d)['title'][:60]}")

    # Sort
    videos = sort_videos(videos, sort_by)

    # Display
    print("\n")
    show_results_html(videos, titles_only=show_titles_only)

    # Store for CSV export
    _last_results = videos

---
## Step 3: Analyze Titles & Generate New Ones

This step does two things:
1. **Analyzes** the scraped titles — shows which patterns, lengths, and words perform best
2. **Generates** new optimized title suggestions for your keyword

**AI-powered generation** (optional but recommended): Get a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey) for much better title suggestions. Without a key, you'll get template-based suggestions instead.

In [ ]:
# =============================================
# STEP 3: ANALYZE & GENERATE TITLES
# =============================================

#@title  { display-mode: "form" }

#@markdown ### AI Title Generation (optional — much better results)
#@markdown Get a **free** API key from [Google AI Studio](https://aistudio.google.com/apikey)
gemini_api_key = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Generate titles for this keyword:
keyword_for_titles = ""  #@param {type:"string"}
number_of_titles = 10  #@param {type:"slider", min:5, max:20, step:5}

import re
from collections import Counter


def detect_title_patterns(title):
    """Detect patterns used in a YouTube title."""
    patterns = []
    t = title.lower()

    if '?' in title:
        patterns.append('Question')
    if 'how to' in t or 'how i' in t:
        patterns.append('How-To')
    if re.search(r'\b\d+\b', title):
        patterns.append('Number/List')

    power = [
        'amazing', 'incredible', 'insane', 'shocking', 'secret', 'ultimate',
        'best', 'worst', 'crazy', 'unbelievable', 'genius', 'hack', 'trick',
        'mistake', 'never', 'always', 'must', 'truth', 'proven', 'powerful',
        'essential', 'complete', 'perfect', 'simple', 'easy', 'free',
    ]
    if any(w in t for w in power):
        patterns.append('Power Words')
    if any(w.isupper() and len(w) > 1 for w in title.split()):
        patterns.append('CAPS Emphasis')
    if '[' in title or '(' in title:
        patterns.append('Brackets')
    if re.search(r'\b20[12]\d\b', title):
        patterns.append('Year Reference')
    if '|' in title or ' - ' in title:
        patterns.append('Separator')
    if not patterns:
        patterns.append('Simple/Direct')
    return patterns


def analyze_titles(videos):
    """Analyze title patterns and their correlation with views."""
    if not videos:
        return None

    # Pattern stats
    pattern_stats = {}
    for v in videos:
        views = v.get('views') or 0
        for p in detect_title_patterns(v['title']):
            if p not in pattern_stats:
                pattern_stats[p] = {'total_views': 0, 'count': 0}
            pattern_stats[p]['total_views'] += views
            pattern_stats[p]['count'] += 1
    for p in pattern_stats:
        pattern_stats[p]['avg_views'] = pattern_stats[p]['total_views'] / pattern_stats[p]['count']

    # Title length buckets
    buckets = {
        'Short (under 40 chars)': (0, 40),
        'Medium (40-60 chars)': (40, 60),
        'Long (60-80 chars)': (60, 80),
        'Very Long (80+ chars)': (80, 999),
    }
    length_stats = {}
    for name, (lo, hi) in buckets.items():
        vlist = [v.get('views') or 0 for v in videos if lo <= len(v['title']) < hi]
        if vlist:
            length_stats[name] = {'avg_views': sum(vlist) / len(vlist), 'count': len(vlist)}

    # Word frequency: top half vs bottom half by views
    sorted_vids = sorted(videos, key=lambda v: v.get('views') or 0, reverse=True)
    mid = max(len(sorted_vids) // 2, 1)
    top_half = sorted_vids[:mid]
    bottom_half = sorted_vids[mid:]

    stop = {
        'the','a','an','in','on','at','to','for','of','and','or','but','is','are',
        'was','were','it','its','i','my','you','your','we','our','this','that',
        'with','from','by','as','not','no','do','if','so','be','he','she','they',
        'will','can','has','have','had','all','me','us','about','just','up','out',
        'what','when','how','why','which','who','where','than','more','most','very',
        'get','got','one','new','like','make','know','don','thing','way',
    }

    def word_freq(vlist):
        words = []
        for v in vlist:
            for w in re.findall(r'[a-zA-Z]+', v['title'].lower()):
                if w not in stop and len(w) > 2:
                    words.append(w)
        return Counter(words)

    return {
        'pattern_stats': pattern_stats,
        'length_stats': length_stats,
        'top_words': word_freq(top_half).most_common(15),
        'bottom_words': word_freq(bottom_half).most_common(15),
        'top_videos': sorted_vids[:5],
        'total': len(videos),
        'avg_len': sum(len(v['title']) for v in videos) / len(videos),
    }


def show_analysis(analysis):
    """Display analysis results as formatted HTML."""
    if not analysis:
        display(HTML("<h3 style='color:red;'>No data to analyze. Run Step 2 first.</h3>"))
        return

    html = "<h2>Title Analysis</h2>"
    html += f"<p>Analyzed <b>{analysis['total']}</b> videos</p>"

    # --- Pattern performance ---
    html += "<h3>Pattern Performance</h3>"
    html += "<p style='color:#666;'>Which title styles get the most views on average?</p>"
    html += "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
    html += "<tr style='background:#f0f0f0;'>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:left;'>Pattern</th>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:right;'>Avg Views</th>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:right;'>Videos</th>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:left;'>Performance</th></tr>"

    sorted_p = sorted(analysis['pattern_stats'].items(), key=lambda x: x[1]['avg_views'], reverse=True)
    mx = sorted_p[0][1]['avg_views'] if sorted_p else 1
    for pat, st in sorted_p:
        bw = int((st['avg_views'] / mx) * 200)
        cl = '#4CAF50' if st['avg_views'] >= mx * 0.7 else '#FF9800' if st['avg_views'] >= mx * 0.4 else '#f44336'
        html += f"<tr>"
        html += f"<td style='padding:6px; border:1px solid #ddd; font-weight:bold;'>{pat}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(int(st['avg_views']))}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{st['count']}</td>"
        html += f"<td style='padding:6px; border:1px solid #ddd;'><div style='background:{cl}; height:16px; width:{bw}px; border-radius:3px;'></div></td>"
        html += f"</tr>"
    html += "</table>"

    # --- Title length ---
    html += "<h3 style='margin-top:20px;'>Title Length Sweet Spot</h3>"
    html += "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
    html += "<tr style='background:#f0f0f0;'>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:left;'>Length</th>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:right;'>Avg Views</th>"
    html += "<th style='padding:8px; border:1px solid #ddd; text-align:right;'>Videos</th></tr>"
    for name in ['Short (under 40 chars)', 'Medium (40-60 chars)', 'Long (60-80 chars)', 'Very Long (80+ chars)']:
        if name in analysis['length_stats']:
            s = analysis['length_stats'][name]
            html += f"<tr><td style='padding:6px; border:1px solid #ddd;'>{name}</td>"
            html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{format_number(int(s['avg_views']))}</td>"
            html += f"<td style='padding:6px; border:1px solid #ddd; text-align:right;'>{s['count']}</td></tr>"
    html += "</table>"
    html += f"<p style='color:#666;'>Average title length: <b>{int(analysis['avg_len'])} characters</b></p>"

    # --- Word comparison ---
    html += "<h3 style='margin-top:20px;'>Common Words: High-View vs Low-View Titles</h3>"
    html += "<div style='display:flex; gap:20px; flex-wrap:wrap;'>"
    html += "<div style='flex:1; min-width:200px;'>"
    html += "<h4 style='color:#4CAF50;'>High-View Titles</h4><div style='font-size:13px;'>"
    for w, c in analysis['top_words']:
        html += f"<span style='background:#e8f5e9; padding:2px 8px; margin:2px; border-radius:12px; display:inline-block;'>{w} ({c})</span>"
    html += "</div></div>"
    html += "<div style='flex:1; min-width:200px;'>"
    html += "<h4 style='color:#f44336;'>Lower-View Titles</h4><div style='font-size:13px;'>"
    for w, c in analysis['bottom_words']:
        html += f"<span style='background:#ffebee; padding:2px 8px; margin:2px; border-radius:12px; display:inline-block;'>{w} ({c})</span>"
    html += "</div></div></div>"

    # --- Top 5 ---
    html += "<h3 style='margin-top:20px;'>Top 5 Best Performing Titles</h3><ol>"
    for v in analysis['top_videos']:
        pats = ', '.join(detect_title_patterns(v['title']))
        html += f"<li style='margin-bottom:8px;'><b>{v['title']}</b><br>"
        html += f"<span style='color:#666; font-size:12px;'>{format_number(v.get('views') or 0)} views | Patterns: {pats}</span></li>"
    html += "</ol>"
    display(HTML(html))


def generate_with_gemini(analysis, keyword, api_key, n=10):
    """Generate optimized titles using Gemini AI."""
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.0-flash')

        sorted_p = sorted(analysis['pattern_stats'].items(), key=lambda x: x[1]['avg_views'], reverse=True)
        best = [p[0] for p in sorted_p[:3]]
        tops = '\n'.join([f"- {v['title']} ({format_number(v.get('views') or 0)} views)" for v in analysis['top_videos']])
        words = ', '.join([w for w, _ in analysis['top_words'][:10]])

        prompt = f"""You are a YouTube title optimization expert. Based on real data analysis, generate {n} optimized title suggestions for a video about "{keyword}".

TOP PERFORMING TITLES FOR THIS TOPIC:
{tops}

DATA-DRIVEN INSIGHTS:
- Best performing title patterns: {', '.join(best)}
- Optimal title length: {int(analysis['avg_len'])} characters
- High-performing words: {words}

RULES:
1. Every title must be about "{keyword}"
2. Use the winning patterns from the data above
3. Keep titles between 40-70 characters
4. Click-worthy but NOT clickbait — be honest and specific
5. Mix different styles: questions, how-tos, lists, direct statements
6. Each title must be unique and distinct from the others

Output ONLY the titles, numbered 1 to {n}. No extra text or explanations."""

        resp = model.generate_content(prompt)
        return resp.text
    except Exception as e:
        return f"Error: {e}"


def generate_pattern_based(analysis, keyword, n=10):
    """Generate titles using template patterns (no API needed)."""
    kw = keyword.strip()
    year = datetime.now().year
    templates = [
        f"How to {kw} - Complete Guide for Beginners",
        f"{kw}: Everything You Need to Know ({year})",
        f"7 {kw} Tips That Actually Work",
        f"Why {kw} Is More Important Than You Think",
        f"The ULTIMATE Guide to {kw} (Step by Step)",
        f"I Tried {kw} for 30 Days - Here's What Happened",
        f"{kw} Explained in 10 Minutes",
        f"Stop Making These {kw} Mistakes",
        f"The Truth About {kw} Nobody Tells You",
        f"{kw} Tutorial: From Zero to Pro",
        f"5 {kw} Secrets the Pros Don't Share",
        f"What Is {kw}? Simple Explanation for Beginners",
        f"BEST {kw} Strategy That Works Every Time",
        f"{kw} for Beginners: Start Here ({year})",
        f"10 Common {kw} Mistakes and How to Fix Them",
        f"Why Most People Fail at {kw}",
        f"{kw} in {year}: What Has Changed?",
        f"Master {kw} With These Simple Steps",
        f"The Only {kw} Guide You Will Ever Need",
        f"{kw}: Beginner to Advanced in One Video",
    ]
    return '\n'.join([f"{i+1}. {t}" for i, t in enumerate(templates[:n])])


# ========== RUN ANALYSIS ==========

try:
    if not _last_results:
        raise NameError
except NameError:
    display(HTML("<h3 style='color:red;'>No scraped results found. Run Step 2 first.</h3>"))
else:
    analysis = analyze_titles(_last_results)
    show_analysis(analysis)

    # --- Title generation ---
    if keyword_for_titles.strip():
        display(HTML("<hr><h2>Generated Title Suggestions</h2>"))
        display(HTML(f"<p>Keyword: <b>{keyword_for_titles}</b></p>"))

        if gemini_api_key.strip():
            import subprocess as _sp
            _sp.run(["pip", "install", "-q", "google-generativeai"], capture_output=True)
            print("Generating AI-powered titles with Gemini...")
            result = generate_with_gemini(analysis, keyword_for_titles, gemini_api_key, number_of_titles)
        else:
            display(HTML(
                "<p style='color:#888;'><i>No API key provided — using template-based generation. "
                "Add a free Gemini API key above for AI-powered titles.</i></p>"
            ))
            result = generate_pattern_based(analysis, keyword_for_titles, number_of_titles)

        if result and not str(result).startswith("Error"):
            html = "<div style='background:#f8f9fa; padding:16px; border-radius:8px; border-left:4px solid #4285f4;'>"
            for line in str(result).strip().split('\n'):
                line = line.strip()
                if line:
                    html += f"<p style='margin:8px 0; font-size:15px;'>{line}</p>"
            html += "</div>"
            display(HTML(html))
        else:
            display(HTML(f"<p style='color:red;'>{result}</p>"))
    else:
        display(HTML(
            "<hr><p style='color:#666;'><b>Tip:</b> Enter a keyword in the form above "
            "and re-run this cell to generate optimized title suggestions based on the analysis.</p>"
        ))

---
## Step 4 (Optional): Download results as a spreadsheet

Run the cell below to download your results as a `.csv` file.  
You can open `.csv` files in **Excel** or **Google Sheets**.

In [ ]:
# =============================================
# STEP 4: DOWNLOAD AS CSV (optional)
# =============================================

#@title Click play to download results as CSV { display-mode: "form" }
filename = "youtube_titles.csv"  #@param {type:"string"}

try:
    if _last_results:
        export_csv_and_download(_last_results, filename)
    else:
        print("No results to export. Run Step 2 first.")
except NameError:
    print("No results to export. Run Step 2 first.")